In [1]:
import quantify_core.data.handling as dh
from qblox_instruments import Cluster, ClusterType
from quantify_scheduler.backends.graph_compilation import SerialCompiler
from quantify_scheduler.device_under_test.quantum_device import QuantumDevice
from quantify_scheduler.device_under_test.transmon_element import BasicTransmonElement
from quantify_core.measurement.control import MeasurementControl
from quantify_scheduler.instrument_coordinator import InstrumentCoordinator
from quantify_scheduler.instrument_coordinator.components.qblox import ClusterComponent
from quantify_core.visualization.pyqt_plotmon import PlotMonitor_pyqt as PlotMonitor
from quantify_scheduler.schedules.schedule import Schedule
from quantify_scheduler.operations.gate_library import Reset
from quantify_scheduler.enums import BinMode
from qcodes.parameters import ManualParameter
from quantify_scheduler.operations.acquisition_library import SSBIntegrationComplex,Trace
from qcodes.instrument import find_or_create_instrument
from quantify_scheduler.operations.pulse_library import SquarePulse, IdlePulse
from quantify_scheduler.gettables import ScheduleGettable
from typing import Callable, Tuple
from numpy import array, arange
meas_datadir = '.data'
dh.set_datadir(meas_datadir)

def get_connected_modules(cluster: Cluster, filter_fn: Callable):
    def checked_filter_fn(mod: ClusterType) -> bool:
        if filter_fn is not None:
            return filter_fn(mod)
        return True

    return {
        mod.slot_idx: mod for mod in cluster.modules if mod.present() and checked_filter_fn(mod)
    }

def configure_measurement_control_loop(
    device: QuantumDevice, cluster: Cluster, live_plotting: bool = False
    ) ->Tuple[MeasurementControl,InstrumentCoordinator]:
    meas_ctrl = find_or_create_instrument(MeasurementControl, recreate=True, name="meas_ctrl")
    ic = find_or_create_instrument(InstrumentCoordinator, recreate=True, name="ic")
    ic.timeout(60*60*120) # 120 hr maximum
    # Add cluster to instrument coordinator
    ic_cluster = ClusterComponent(cluster)
    ic.add_component(ic_cluster)

    if live_plotting:
        # Associate plot monitor with measurement controller
        plotmon = find_or_create_instrument(PlotMonitor, recreate=False, name="PlotMonitor")
        meas_ctrl.instr_plotmon(plotmon.name)

    # Associate measurement controller and instrument coordinator with the quantum device
    device.instr_measurement_control(meas_ctrl.name)
    device.instr_instrument_coordinator(ic.name)

    return (meas_ctrl, ic)

def pulse_preview(quantum_device:QuantumDevice,sche_func:Schedule, sche_kwargs:dict, **kwargs):
    import plotly.io as pio
    pio.renderers.default='browser'
    device_compiler = SerialCompiler("Device compiler", quantum_device)
    comp_sched = device_compiler.compile(
        sche_func(**sche_kwargs)
    )
    comp_sched.plot_pulse_diagram(plot_backend="plotly",**kwargs).show() 

def Integration(sche,q,R_inte_delay:float,R_inte_duration,ref_pulse_sche,acq_index,acq_channel:int=0,single_shot:bool=False,get_trace:bool=False,trace_recordlength:float=5*1e-6):
    if single_shot== False:     
        bin_mode=BinMode.AVERAGE
    else: bin_mode=BinMode.APPEND
    # Trace acquisition does not support APPEND bin mode !!!
    if get_trace==False:
        return sche.add(SSBIntegrationComplex(
            duration=R_inte_duration[q],
            port="q:res",
            clock=q+".ro",
            acq_index=acq_index,
            acq_channel=acq_channel,
            bin_mode=bin_mode,
            ),rel_time=R_inte_delay
            ,ref_op=ref_pulse_sche,ref_pt="start")
    else:  
        return sche.add(Trace(
                duration=trace_recordlength,
                port="q:res",
                clock=q+".ro",
                acq_index=acq_index,
                acq_channel=acq_channel,
                bin_mode=BinMode.AVERAGE,
                ),rel_time=R_inte_delay
                ,ref_op=ref_pulse_sche,ref_pt="start")


In [3]:
Hcfg = {
    "backend": "quantify_scheduler.backends.qblox_backend.hardware_compile",
    f"clusterdr1": {
        "sequence_to_file": False,  # Boolean flag which dumps waveforms and program dict to JSON file
        "ref": "internal",  # Use shared clock reference of the cluster
        "instrument_type": "Cluster",
        # ============ DRIVE ============#
        f"clusterdr1_module4": {
            "instrument_type": "QCM_RF",
            "complex_output_0": {
                "output_att": 0,
                "dc_mixer_offset_I": 0.0,
                "dc_mixer_offset_Q": 0.0,
                "lo_freq": 4e9,
                "portclock_configs": [
                    {
                        "port": "q0:mw",
                        "clock": "q0.01",
                        "mixer_amp_ratio": 1.0,
                        "mixer_phase_error_deg": 0.0,
                    }
                ],
            },
        },
        # ============ FLUX ============#
        f"clusterdr1_module2": {
            "instrument_type": "QCM",
            "real_output_0": {"portclock_configs": [{"port": "q0:fl", "clock": "cl0.baseband","interm_freq":0}]},
        },
        # ============ READOUT ============#
        f"clusterdr1_module6": {
            "instrument_type": "QRM_RF",
            "complex_output_0": {
                "output_att": 0,
                "input_att": 0,
                "dc_mixer_offset_I": 0.0,
                "dc_mixer_offset_Q": 0.0,
                "lo_freq": 5.9e9,       # *** Should be set as a parameter later on
                "portclock_configs": [
                    {
                        "port": "q:res",
                        "clock": "q0.ro",
                        "mixer_amp_ratio": 1.0,
                        "mixer_phase_error_deg": 0.0,
                    },
                    
                ],
            },
        },
    },
}

In [4]:
""" Connect cluster, create QuantumDevice, get MeasurementControl and InstrumentCoordinator"""
target_q = 'q0'
cluster = Cluster(name = "clusterdr1",identifier = f"qum.phys.sinica.edu.tw", port=5011)
cluster.reset()

QRM_RFs = get_connected_modules(cluster,lambda mod: mod.is_qrm_type and mod.is_rf_type)
for slot_idx in QRM_RFs:
    for i in range(6):
        getattr(QRM_RFs[slot_idx], f"sequencer{i}").nco_prop_delay_comp_en(True)      
        getattr(QRM_RFs[slot_idx], f"sequencer{i}").nco_prop_delay_comp(50)

quantum_device = find_or_create_instrument(QuantumDevice, recreate=True, name="QPUdr1")
quantum_device.hardware_config(Hcfg)
for qb_idx in range(1):
    qubit = find_or_create_instrument(BasicTransmonElement, recreate=True, name=f"q{qb_idx}")
    qubit.measure.acq_channel(qb_idx)
    quantum_device.add_element(qubit)

meas_ctrl, ic = configure_measurement_control_loop(quantum_device, cluster)

Fctrl: callable = {
            "q0":cluster.module2.out0_offset,
        }

Fctrl['q0'](0.0)


In [4]:
""" Pulse schedule """

def Z(sche:Schedule,Z_amp,Du,q):
    return sche.add(SquarePulse(duration= Du,amp=Z_amp, port=q+":fl", clock="cl0.baseband"))
    

def Z_gate_demo_sche(
    q:str,
    z_duration:float,
    Z_amp:any, 
    repetitions:int=1,
) -> Schedule:
    sched = Schedule("Zgate_demo",repetitions=repetitions)
    
    for idx, z in enumerate(Z_amp): 
        sched.add(Reset(q))

        Z(sched,z,z_duration,q)
        
    return sched

In [5]:
def Zgate_demo_spec(meas_ctrl:MeasurementControl,z_amplitude:float,n_avg:int=1000,run:bool=True,q:str='q1'):
    
    sche_func = Z_gate_demo_sche

    qubit_info = quantum_device.get_element(q)
    z_amp = array([z_amplitude]*100)
    z_dura = 100e-6
    Z_bias = ManualParameter(name="Z", unit="V", label="Z bias")
    Z_bias.batched = True
    qubit_info.reset.duration(100e-6)
 
    spec_sched_kwargs = dict(   
        q=q,
        Z_amp=z_amp,
        z_duration=z_dura,
    )
    
    if run:
        gettable = ScheduleGettable(
            quantum_device,
            schedule_function=sche_func, 
            schedule_kwargs=spec_sched_kwargs,
            real_imag=False,
            batched=True,
        )
        quantum_device.cfg_sched_repetitions(n_avg)
        meas_ctrl.gettables(gettable)
        meas_ctrl.settables(Z_bias)
        meas_ctrl.setpoints(arange(100))
        qs_ds = meas_ctrl.run("Zgate_demo")
        
        
    else:
        para = array(z_amp)
        spec_sched_kwargs['Z_amp']= para.reshape(para.shape or (1,))
        pulse_preview(quantum_device,sche_func,spec_sched_kwargs)
        

In [ ]:

z_offset = 0.0
z_amplitude = 0.1 # V
# when z_amplitude = 0.10, the real amplitude observed from oscilloscope is 0.1755
# when z_amplitude = 0.05, the real amplitude observed from oscilloscope is 0.0877

Fctrl[target_q](z_offset)
Zgate_demo_spec(meas_ctrl,z_amplitude,n_avg=50000,run=True,q=target_q)

Starting batched measurement...
Iterative settable(s) [outer loop(s)]:
	 --- (None) --- 
Batched settable(s):
	 Z 
Batch size limit: 100



[!!!] 1 interruption(s) signaled. Stopping after this iteration/batch.
[Send 4 more interruptions to forcestop (not safe!)].



In [5]:
Fctrl[target_q](0.0)
cluster.reset() 
meas_ctrl.close_all() 